In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from ML_xgboost import train_xgboost_model
from ML_lightgbm import train_lightgbm_model
from ML_catboost import train_catboost_model
from ML_randomforest import train_randomforest_model 

df_model = pd.read_csv("D:/1.SJTU/MentalHealthProjects/ML_CLPN_Project/01_Input/df_model.csv")

selected_vars = [
    "somatic_y1", "BMI_T1_cat", "sleep_dura_T1_cat", "sleep_quali_T1", "insomnia_y1",
    "life_satis_y1", "ms_ses_y1", "per_stress_y1", "ms_stress_y1", "depression_y1", "anxiety_y1",
    "ace", "loneliness_y1", "support_y1", "gender_T1", "age_T1", "residence", "income", 
     "income_ineqCity_y1", "sss_now", "marrige_par_bin", "edu_pa",
    "eat_unctl_y1", "eat_emot_y1", "food_sweetdrink_T1", "food_takeout_T1",
    "IPAQ_T1_1_bin", "IPAQ_T1_3_bin", "IPAQ_T1_5_bin", "screenT_weekday_T1", "screenT_weekend_T1",
    "psmu_y1", "media_BadMood_T1", "media_GoodMood_T1", "edu_self","grade_T1"
]
y_col = "depression_y2"
y_cont = pd.to_numeric(df_model[y_col], errors="coerce")
# 二分类标签：PHQ-9 ≥ 5
y = (y_cont >= 5).astype(int)
X = df_model[selected_vars].copy()

RANDOM_STATE = 20516
TEST_SIZE = 0.20
VAL_SIZE = 0.20

# === 1. Training XGBoost ===
print("=== 1. Training XGBoost ===")
res_xgb = train_xgboost_model(X, y, selected_vars, RANDOM_STATE, TEST_SIZE, VAL_SIZE)
print(f"XGB Test AUC: {res_xgb['test_auc']:.4f}")

# === 2. Training LightGBM ===
print("\n=== 2. Training LightGBM ===")
res_lgbm = train_lightgbm_model(X, y, selected_vars, RANDOM_STATE, TEST_SIZE, VAL_SIZE)
print(f"LGBM Test AUC: {res_lgbm['test_auc']:.4f}")

# === 3. Training CatBoost ===
print("\n=== 3. Training CatBoost ===")
res_cat = train_catboost_model(X, y, selected_vars, RANDOM_STATE, TEST_SIZE, VAL_SIZE)
print(f"CatBoost Test AUC: {res_cat['test_auc']:.4f}")

# === 4. Training Random Forest ===
print("\n=== 4. Training RandomForest ===")
res_rf = train_randomforest_model(X, y, selected_vars, RANDOM_STATE, TEST_SIZE, VAL_SIZE)
print(f"RF Test AUC: {res_rf['test_auc']:.4f}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

models = ['RandomForest', 'XGBoost', 'LightGBM', 'CatBoost']

# AUC Score: higher is better
auc_scores = [
    res_rf['test_auc'],
    res_xgb['test_auc'],
    res_lgbm['test_auc'],
    res_cat['test_auc']
]

# F1 Score: higher is better (threshold-dependent)
f1_scores = [
    res_rf['test_f1'],
    res_xgb['test_f1'],
    res_lgbm['test_f1'],
    res_cat['test_f1']
]

# ACC Score: higher is better (threshold-dependent)
acc_scores = [
    res_rf['test_acc'],
    res_xgb['test_acc'],
    res_lgbm['test_acc'],
    res_cat['test_acc']
]

# create DataFrame
df_perf = pd.DataFrame({
    'Model': models,
    'Test AUC': auc_scores,
    'Test F1': f1_scores,
    'Test Accuracy': acc_scores
})

plt.rcParams.update({
    'font.family': 'Times New Roman',
    'axes.titlesize': 22,
    'axes.labelsize': 20,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 18,
    'font.size': 18
})

fig, axes = plt.subplots(1, 3, figsize=(22, 8), dpi=600, tight_layout=True)

# AUC plot
sns.barplot(x='Model', y='Test AUC', data=df_perf, ax=axes[0], palette="Greens")
axes[0].set_title(f'Test Set AUC Comparison')
axes[0].set_ylabel('AUC')
axes[0].tick_params(axis='x', labelsize=16) # rotation=15
axes[0].set_ylim(0.5, 1)
for i in axes[0].containers:
    axes[0].bar_label(i, fmt='%.4f', label_type='edge', padding=4, fontsize=14)
axes[0].margins(y=0.1)
axes[0].legend(
    loc='upper right',
    bbox_to_anchor=(1, 1),
    framealpha=0.7     # 半透明
)
# F1 plot
sns.barplot(x='Model', y='Test F1', data=df_perf, ax=axes[1], palette="Oranges")
axes[1].set_title(f'Test Set F1-score Comparison')
axes[1].set_ylabel('F1-score')
axes[1].tick_params(axis='x', labelsize=16)
axes[1].set_ylim(0.5, 1)
for i in axes[1].containers:
    axes[1].bar_label(i, fmt='%.4f', label_type='edge', padding=4, fontsize=14)
axes[1].margins(y=0.1)
axes[1].legend(
    loc='upper right',
    bbox_to_anchor=(1, 1),
    framealpha=0.7     # 半透明
)

# ACC plot
sns.barplot(x='Model', y='Test Accuracy', data=df_perf, ax=axes[2], palette="Purples")
axes[2].set_title('Test Set Accuracy Comparison')
axes[2].set_ylabel('Accuracy')
axes[2].set_ylim(0.5, 1)
axes[2].tick_params(axis='x', labelsize=16)
axes[2].legend(
    loc='upper right',
    bbox_to_anchor=(1, 1),
    framealpha=0.7     # 半透明
)
for bar in axes[2].containers:
    axes[2].bar_label(bar, fmt='%.4f', label_type='edge', padding=4, fontsize=14)
axes[2].margins(y=0.1)


os.makedirs("ml_dep/plots_dep_cat", exist_ok=True)
save_path = "ml_dep/plots_dep_cat/Depressive_Performance_1x3.png"
plt.savefig(save_path, dpi=600, bbox_inches='tight')
plt.close()

print(f"Saved to: {save_path}")

In [ ]:
results = {
    'CatBoost': res_cat,
    'LGBM': res_lgbm,
    'RF': res_rf,
    'XGBoost': res_xgb
}

data_list = []

for model_name, res in results.items():
    # Train set metrics
    data_list.append({
        'Model': model_name,
        'Dataset': 'Train',
        'AUC': res['train_auc'],
        'F1': res['train_f1'],
        'Accuracy': res['train_acc']
    })
    
    # Evalidation set metrics
    data_list.append({
        'Model': model_name,
        'Dataset': 'Validation',
        'AUC': res['val_auc'],
        'F1': res['val_f1'],
        'Accuracy': res['val_acc']
    })
    
    # Test set metrics
    data_list.append({
        'Model': model_name,
        'Dataset': 'Test',
        'AUC': res['test_auc'],
        'F1': res['test_f1'],
        'Accuracy': res['test_acc']
    })

# DataFrame
df_final_perf = pd.DataFrame(data_list)

os.makedirs("ml_dep/output", exist_ok=True) 

excel_save_path = f"ml_dep/output/Single_Model_Full_Performance.xlsx"
df_final_perf = df_final_perf.sort_values(by=['Model', 'Dataset'])

df_final_perf.to_excel(excel_save_path, index=False)

print("\n" + "="*50)
print(f"All model performances had been saved in: {excel_save_path}")
print("="*50)
print("\n=== The comparison of model performance ===")
print(df_final_perf)

# === Plot ===
plt.rcParams.update({
    'font.family': 'Times New Roman',
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 18,
    'font.size': 18
})

plt.rcParams['font.family'] = 'Times New Roman'

fig, axes = plt.subplots(1, 3, figsize=(18, 7), dpi=600, tight_layout=True)

# --- 1. AUC ---
sns.barplot(
    x='Model', 
    y='AUC', 
    hue='Dataset', 
    data=df_final_perf, 
    ax=axes[0], 
    palette="Greens"
)
axes[0].set_ylim(0, 1)
axes[0].set_title('AUC Performance Comparison')
axes[0].set_ylabel('AUC Score')
axes[0].tick_params(axis='x')
axes[0].legend(
        loc = 'upper center',
        bbox_to_anchor=(0.5, 0.9995),
        ncol=3,
        frameon=True,
        fontsize=15.5,
        framealpha=0.7     # 半透明
)

# --- 2. F1 Score ---
sns.barplot(
    x='Model', 
    y='F1', 
    hue='Dataset', 
    data=df_final_perf, 
    ax=axes[1], 
    palette="Oranges"
)
axes[1].set_ylim(0, 1)
axes[1].set_title('F1 Score Performance Comparison')
axes[1].set_ylabel('F1 Score')
axes[1].tick_params(axis='x')
axes[1].legend(
        loc = 'upper center',
        bbox_to_anchor=(0.5, 0.9995),
        ncol=3,
        frameon=True,
        fontsize=15.5,
        framealpha=0.7     # 半透明
)

# --- 3. Accuracy ---
sns.barplot(
    x='Model', 
    y='Accuracy', 
    hue='Dataset', 
    data=df_final_perf, 
    ax=axes[2], 
    palette="Purples"
)
axes[2].set_ylim(0, 1)
axes[2].set_title('Accuracy Performance Comparison')
axes[2].set_ylabel('Accuracy Score')
axes[2].tick_params(axis='x')
axes[2].legend(
        loc = 'upper center',
        bbox_to_anchor=(0.5, 0.9995),
        ncol=3,
        frameon=True,
        fontsize=15.5,
        framealpha=0.7     # 半透明
)

# plt.suptitle(f'Single Model Performance Across Train, Validation, and Test Sets', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])

png_save_path = f"ml_dep/plots_dep_cat/Performance_Comparison_5Models_Full_Comparison.png"
plt.savefig(png_save_path)
plt.close()

print(f"\n All performance comparison had been save in: {png_save_path}")

In [ ]:
import os
import pandas as pd
import shap
import matplotlib.pyplot as plt
from ML_shap_dep import run_full_shap_analysis

VARIABLE_MAPPING = {
    "somatic_y1": "Somatic Symptoms",
    "BMI_T1_cat": "BMI Category", 
    "sleep_dura_T1_cat": "Sleep Duration",
    "sleep_quali_T1": "Sleep Quality",
    "insomnia_y1": "Insomnia",
    
    "life_satis_y1": "Life Satisfaction",
    "ms_ses_y1": "Mindset of Socioeconomic Status", 
    "per_stress_y1": "Perceived Stress", 
    "ms_stress_y1": "Stress Mindset",
    "depression_y1": "Baseline Depressive Symptoms",
    "anxiety_y1": "BaselineAnxiety Symptoms",
    
    "ace": "Adverse Childhood Experiences",
    
    "loneliness_y1": "Loneliness", 
    "support_y1": "Social Support",
    
    "gender_T1": "Gender",
    "age_T1": "Age",
    "grade_T1": "Grade",
    "residence": "Residence", 
    "income": "Household Income",
    "income_ineqCity_y1": "Perceived Income Inequality",
    "sss_now": "Subjective Socialeconomic Status",
    "marrige_par_bin": "Parental Marital Status",
    "edu_pa": "Parental Educational Level",
    
    "eat_unctl_y1": "Uncontrolled Eating",
    "eat_emot_y1": "Emotional Eating", 
    "food_sweetdrink_T1": "Sugar-Sweetened Beverage Consumption",
    "food_takeout_T1": "Takeaway Frequency",
    
    "IPAQ_T1_1_bin": "Vigorous Physical Activity",
    "IPAQ_T1_3_bin": "Moderate Physical Activity", 
    "IPAQ_T1_5_bin": "Walking Activity",
    
    "screenT_weekday_T1": "Weekday Screen Time",
    "screenT_weekend_T1": "Weekend Screen Time",
    
    "psmu_y1": "Problematic Social Media Use",
    "media_BadMood_T1": "Social Media Posting of Negative Emotion", 
    "media_GoodMood_T1": "Social Media Posting of Positive Emotion",
    
    "edu_self": "Educational Aspiration"
}


all_results = {
    'RandomForest': res_rf,
    'XGBoost': res_xgb,
    'LightGBM': res_lgbm,
    'CatBoost': res_cat
}


FLAG_SHOW = False 
FLAG_TITLE = False 
IS_LOG = False 
TOPN = 15


for name, results in all_results.items():
    run_full_shap_analysis(
        model_name=name,
        results=results,
        df_model=df_model,         
        selected_vars=selected_vars,
        variable_mapping=VARIABLE_MAPPING,
        is_log_transformed=IS_LOG,
        top_n=TOPN,
        flag_show=FLAG_SHOW,
        flag_title=FLAG_TITLE
    )